## Step 0 — Verify GPU and environment

In [ ]:
import subprocess
import os

# Check GPU
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
    capture_output=True,
    text=True
)

print(
    gpu_check.stdout
    if gpu_check.returncode == 0
    else "No GPU detected — enable GPU in Settings before continuing."
)

# Check Kaggle input files
print("\nContents of /kaggle/input/")

if os.path.exists("/kaggle/input"):
    for item in os.listdir("/kaggle/input"):
        print(" ", item)
else:
    print("  (none found — attach your dataset via 'Add Input' first)")

## Step 1 — Install pinned dependencies

Installed once, no reinstall later in the notebook. The original notebook
uninstalled and reinstalled these same packages a second time in a later
cell (for the now-removed diffusers alternative pipeline) — that redundancy
is gone here.

**After this cell finishes, restart the session** (Kaggle: Run → Restart
Session), then continue from Step 2. This matches the original notebook's
requirement and is still necessary — some of these packages don't take
effect in the same running kernel that installed them.

In [ ]:
import subprocess, sys
python = sys.executable

print("Installing pinned dependencies for kohya_ss compatibility...")

subprocess.run([python, "-m", "pip", "uninstall", "-y",
    "diffusers", "transformers", "huggingface-hub", "accelerate", "peft"])

subprocess.run([python, "-m", "pip", "install", "-q",
    "diffusers==0.25.1",
    "transformers==4.38.2",
    "huggingface-hub==0.21.4",
    "accelerate==0.27.2",
    "peft==0.9.0",
    "safetensors>=0.4.0",
    "bitsandbytes>=0.41.0",
    "xformers",
    "einops",
    "omegaconf",
    "toml",
    "ftfy",
    "albumentations",
    "timm",
    "voluptuous",
])

print("\nDone. NOW RESTART THE SESSION (Run -> Restart Session).")
print("After restarting, skip this cell and continue from Step 2.")

## Step 2 — Clone kohya_ss (sd-scripts)

In [1]:
import os, subprocess

KOHYA_DIR = "/kaggle/working/kohya_ss"

if os.path.exists(KOHYA_DIR):
    subprocess.run(["rm", "-rf", KOHYA_DIR])
    print("Removed old kohya_ss clone")

print("Cloning kohya_ss...")
result = subprocess.run([
    "git", "clone", "-q", "--depth", "1", "-b", "v23.1.6",
    "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR
])

if result.returncode != 0:
    print("Tagged version failed, cloning main...")
    subprocess.run(["git", "clone", "-q", "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR])

trainer = os.path.join(KOHYA_DIR, "sdxl_train_network.py")
if os.path.exists(trainer):
    print(f"Trainer found: {trainer}")
else:
    print(f"Trainer NOT found at {trainer} -- check the clone above")

Removed old kohya_ss clone
Cloning kohya_ss...


fatal: Remote branch v23.1.6 not found in upstream origin


Tagged version failed, cloning main...
Trainer found: /kaggle/working/kohya_ss/sdxl_train_network.py


## Step 2b — Install kohya_ss requirements and verify imports

In [2]:
import subprocess, sys, os

python = sys.executable
KOHYA_DIR = "/kaggle/working/kohya_ss"
req_file = os.path.join(KOHYA_DIR, "requirements.txt")

if os.path.exists(req_file):
    print("Installing kohya_ss requirements...")
    subprocess.run([python, "-m", "pip", "install", "-q", "-r", req_file])
else:
    print("requirements.txt not found -- skipping (already installed in Step 1)")

print("\nVerifying imports...")
try:
    import diffusers, transformers, accelerate
    print(f"  diffusers   : {diffusers.__version__}")
    print(f"  transformers: {transformers.__version__}")
    print(f"  accelerate  : {accelerate.__version__}")
    print("All imports OK")
except ImportError as e:
    print(f"Import error: {e}")
    print("Restart the session and re-run from Step 1.")

Installing kohya_ss requirements...


ERROR: file:///kaggle/working (from -r /kaggle/working/kohya_ss/requirements.txt (line 49)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.



Verifying imports...
  diffusers   : 0.25.1
  transformers: 4.38.2
  accelerate  : 0.27.2
All imports OK


## Step 3 — Download SDXL base model

Pulled directly from Hugging Face rather than uploaded as a Kaggle Dataset
-- reproducible without a manual multi-GB upload step, and cached so
re-running this cell doesn't re-download.

In [3]:
from huggingface_hub import hf_hub_download
import shutil, os

MODEL_LOCAL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

if not os.path.exists(MODEL_LOCAL_PATH):
    print("Downloading SDXL base model (this can take a while)...")
    downloaded_path = hf_hub_download(
        repo_id="stabilityai/stable-diffusion-xl-base-1.0",
        filename="sd_xl_base_1.0.safetensors",
    )
    shutil.copy(downloaded_path, MODEL_LOCAL_PATH)
    print(f"Model saved to {MODEL_LOCAL_PATH}")
else:
    print(f"Model already present at {MODEL_LOCAL_PATH}")

Model already present at /kaggle/working/sd_xl_base_1.0.safetensors


## Step 4 — Configure run and verify dataset

Set `RUN = "a"` for clean images or `RUN = "b"` for cloaked images, then
run this cell. **This is the single place that controls which dataset gets
trained** -- the training cell in Step 6 reads these variables directly,
rather than hardcoding its own copy of them (which is what the original
notebook's Run A / Run B cells did, making the `RUN` toggle here have no
actual effect on them).

Change `RUN` and re-run this cell, then re-run Step 6, to switch between
Run A and Run B -- no need to duplicate the training cell.

In [4]:
import os

RUN = "a"   # "a" for run-a0-1, change to "b" if you attach run-b

# ============================================================
# KAGGLE PATHS
# ============================================================

INPUT_BASE = "/kaggle/input/datasets/leothiii"

DATASET_PATH = os.path.join(
    INPUT_BASE,
    "runa-05"
)

MODEL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

OUTPUT_DIR = f"/kaggle/working/outputs/lora_run_{RUN}"
OUTPUT_NAME = f"lora_run_{RUN}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# CHECK
# ============================================================

print(f"RUN        : {RUN.upper()}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Output dir : {OUTPUT_DIR}")

if os.path.exists(DATASET_PATH):
    print("\n✅ Dataset path exists!")

    print("\nContents:")
    for item in os.listdir(DATASET_PATH):
        print(" ", item)
else:
    print("\n❌ Dataset path NOT FOUND")

RUN        : A
Dataset    : /kaggle/input/datasets/leothiii/runa-05
Output dir : /kaggle/working/outputs/lora_run_a

✅ Dataset path exists!

Contents:
  30_person


##### Step 5 — Run LoRA training

Single parameterized cell -- reads `RUN`, `DATASET_PATH`, `MODEL_PATH`,
`OUTPUT_DIR`, `OUTPUT_NAME` from Step 4 (and Step 5, if you ran it). To
train Run B after Run A: go back to Step 4, set `RUN = "b"`, re-run Step 4
(and Step 5 if used), then re-run this cell -- no separate duplicated cell
needed, unlike the original notebook's Run A/Run B cells which hardcoded
their own copies of these variables independently of the `RUN` toggle.

**Parameters** (matches Table 5 of the thesis): Network Dim 16, Alpha 8,
Steps 500, LR 1e-4, Resolution 512x512, Optimizer AdamW8bit, seed 42.

In [ ]:
# ================================================================
# SDXL LoRA TRAINING - KAGGLE / TESLA T4
# ================================================================

import os
import subprocess
import sys

# ================================================================
# CONFIGURATION
# ================================================================

KOHYA_DIR = "/kaggle/working/kohya_ss"
TRAIN_SCRIPT = f"{KOHYA_DIR}/sdxl_train_network.py"

MODEL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

# IMPORTANT:
# The dataset is read-only under /kaggle/input.
TRAIN_DATA_DIR = "/kaggle/input/datasets/leothiii/runa05"

OUTPUT_DIR = "/kaggle/working/outputs/lora_run_a"
OUTPUT_NAME = "lora_runa05"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ================================================================
# CHECK REQUIRED FILES
# ================================================================

print("=" * 70)
print("CHECKING FILES")
print("=" * 70)

required_paths = {
    "Kohya directory": KOHYA_DIR,
    "Training script": TRAIN_SCRIPT,
    "Base SDXL model": MODEL_PATH,
    "Training directory": TRAIN_DATA_DIR,
}

for name, path in required_paths.items():
    if os.path.exists(path):
        print(f"OK       {name}: {path}")
    else:
        print(f"ERROR    {name}: {path}")
        raise FileNotFoundError(
            f"{name} was not found:\n{path}"
        )


# ================================================================
# CHECK DATASET
# ================================================================

print()
print("=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print(TRAIN_DATA_DIR)

for root, dirs, files in os.walk(TRAIN_DATA_DIR):
    level = root.replace(TRAIN_DATA_DIR, "").count(os.sep)

    # Only display the important levels
    if level <= 1:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

        for filename in sorted(files)[:10]:
            print(f"{indent}    {filename}")

        if len(files) > 10:
            print(f"{indent}    ... ({len(files)} files total)")


# ================================================================
# VERIFY 10_PERSON FOLDER
# ================================================================

expected_dataset_folder = os.path.join(
    TRAIN_DATA_DIR,
    "10_person"
)

if os.path.isdir(expected_dataset_folder):
    print()
    print("OK       Found 10_person dataset folder")
else:
    print()
    print("WARNING  10_person folder was not found:")
    print(f"         {expected_dataset_folder}")
    print()
    print("Kohya requires the repeat folder to be directly")
    print("inside TRAIN_DATA_DIR.")
    raise FileNotFoundError(expected_dataset_folder)


# ================================================================
# COUNT IMAGES / CAPTIONS
# ================================================================

image_extensions = (
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
)

images = [
    f for f in os.listdir(expected_dataset_folder)
    if f.lower().endswith(image_extensions)
]

captions = [
    f for f in os.listdir(expected_dataset_folder)
    if f.lower().endswith(".txt")
]

print()
print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print(f"Images:   {len(images)}")
print(f"Captions: {len(captions)}")

if len(images) == 0:
    raise RuntimeError("No training images found.")

if len(images) != len(captions):
    print()
    print("WARNING:")
    print("The number of images and .txt captions does not match.")
    print("Kohya may train some images without captions.")
else:
    print("OK       Every image has a .txt caption")


# ================================================================
# TRAINING COMMAND
# ================================================================

command = [
    sys.executable,

    # Accelerate
    "-m",
    "accelerate.commands.launch",
    "--num_processes=1",
    "--num_machines=1",
    "--mixed_precision=fp16",

    # Kohya training script
    TRAIN_SCRIPT,

    # ------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------
    f"--pretrained_model_name_or_path={MODEL_PATH}",

    # ------------------------------------------------------------
    # DATASET
    # ------------------------------------------------------------
    f"--train_data_dir={TRAIN_DATA_DIR}",

    # ------------------------------------------------------------
    # OUTPUT
    # ------------------------------------------------------------
    f"--output_dir={OUTPUT_DIR}",
    f"--output_name={OUTPUT_NAME}",
    "--save_model_as=safetensors",

    # ------------------------------------------------------------
    # RESOLUTION
    # ------------------------------------------------------------
    "--resolution=512,512",

    # ------------------------------------------------------------
    # LoRA
    # ------------------------------------------------------------
    "--network_module=networks.lora",
    "--network_dim=32",
    "--network_alpha=16",

    # ------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------
    "--train_batch_size=1",
    "--max_train_epochs=10",

    # ------------------------------------------------------------
    # LEARNING RATES
    # ------------------------------------------------------------
    "--learning_rate=1e-4",
    "--unet_lr=1e-4",
    "--text_encoder_lr=5e-5",

    # ------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------
    "--optimizer_type=AdamW8bit",

    # ------------------------------------------------------------
    # PRECISION
    # ------------------------------------------------------------
    "--mixed_precision=fp16",
    "--save_precision=fp16",

    # ------------------------------------------------------------
    # MEMORY OPTIMIZATION
    # ------------------------------------------------------------
    "--gradient_checkpointing",
    "--sdpa",

    # ------------------------------------------------------------
    # LATENT CACHE
    #
    # IMPORTANT:
    # Do NOT use --cache_latents_to_disk because the dataset
    # lives under /kaggle/input, which is read-only.
    # ------------------------------------------------------------
    "--cache_latents",
    "--vae_batch_size=1",

    # ------------------------------------------------------------
    # CAPTIONS
    # ------------------------------------------------------------
    "--caption_extension=.txt",
    "--shuffle_caption",
    "--keep_tokens=1",

    # ------------------------------------------------------------
    # SAVING
    # ------------------------------------------------------------
    "--save_every_n_epochs=1",

    # ------------------------------------------------------------
    # REPRODUCIBILITY
    # ------------------------------------------------------------
    "--seed=42",
]


# ================================================================
# DISPLAY COMMAND
# ================================================================

print()
print("=" * 70)
print("TRAINING COMMAND")
print("=" * 70)

print(" ".join(command))


# ================================================================
# START TRAINING
# ================================================================

print()
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print()

process = subprocess.Popen(
    command,
    cwd=KOHYA_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream Kohya output live
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()


# ================================================================
# TRAINING RESULT
# ================================================================

print()
print("=" * 70)

if return_code == 0:
    print("✓ TRAINING COMPLETED SUCCESSFULLY")
else:
    print("✗ TRAINING FAILED")
    print(f"Exit code: {return_code}")

print("=" * 70)


# ================================================================
# SHOW OUTPUT FILES
# ================================================================

print()
print("OUTPUT FILES")
print("=" * 70)

found_output = False

if os.path.exists(OUTPUT_DIR):

    for root, dirs, files in os.walk(OUTPUT_DIR):

        for filename in sorted(files):

            filepath = os.path.join(root, filename)

            print(filepath)

            found_output = True

if not found_output:
    print("No output files found.")


# ================================================================
# FINAL STATUS
# ================================================================

print()
print("=" * 70)

if return_code == 0:
    print("DONE")
    print()
    print(f"LoRA output directory:")
    print(OUTPUT_DIR)
else:
    print("TRAINING DID NOT COMPLETE.")
    print()
    print("The error above is the important part.")
    print("Copy the traceback if another error appears.")

print("=" * 70)


CHECKING FILES
OK       Kohya directory: /kaggle/working/kohya_ss
OK       Training script: /kaggle/working/kohya_ss/sdxl_train_network.py
OK       Base SDXL model: /kaggle/working/sd_xl_base_1.0.safetensors
OK       Training directory: /kaggle/input/datasets/leothiii/runa05

DATASET STRUCTURE
/kaggle/input/datasets/leothiii/runa05
runa05/
    10_person/
        img1.png
        img1.txt
        img10.png
        img10.txt
        img11.png
        img11.txt
        img12.png
        img12.txt
        img13.png
        img13.txt
        ... (60 files total)

OK       Found 10_person dataset folder

DATASET CHECK
Images:   30
Captions: 30
OK       Every image has a .txt caption

TRAINING COMMAND
/usr/bin/python3 -m accelerate.commands.launch --num_processes=1 --num_machines=1 --mixed_precision=fp16 /kaggle/working/kohya_ss/sdxl_train_network.py --pretrained_model_name_or_path=/kaggle/working/sd_xl_base_1.0.safetensors --train_data_dir=/kaggle/input/datasets/leothiii/runa05 --output_dir=